# 09 · M1.5 — M1.3 with an EWMA sd (halflife 63)

## Executive summary (read this first)

**M1.5 = M1.3 with one change: the sd of each asset is an EWMA sd** instead of the plain sd of the 260 steps. A step's weight
halves every 63 steps (`weight = 0.5 ** (age ÷ 63)`, over the last 8 × 63 = 504 steps), so the width follows the latest
months more closely. Everything else is M1.3:

| piece | M1.3 | **M1.5** |
|---|---|---|
| transform | per asset type | same |
| centre | drift rule (drift for CPI and NFP only) | same, drift from the 260 steps |
| sd (width) | plain sd of the last 260 steps | **EWMA sd, halflife 63** (last 504 steps) |
| correlation between assets | the 260 steps | same |
| shocks | Student-t, ν fitted from the 260 steps' kurtosis (rates, FX, equity); normal for macro | same |
| widen | 1.0 | 1.0 |

The covariance is `correlation_ij × EWMA sd_i × EWMA sd_j`. Monthly units use the same rule (halflife 63 months). Nothing is
tuned: the halflife 63 is set in advance (it is production's F4 value).

**The test:** notebook 01's backtest dates and answers; competition score per unit vs M0. Below 1 beats M0.

Results: §2 (every model side by side), §3 (per family and asset type), §5 (in short). No realized outcome is read.

## 0 — Setup (as notebook 07)

In [1]:
import bisect, multiprocessing as mp, pathlib, sys, time, tomllib, warnings, zlib
import numpy as np, pandas as pd, pyarrow.parquet as pq

HERE = pathlib.Path.cwd() if pathlib.Path.cwd().name == "model_experiment_v3" else pathlib.Path.cwd() / "model_experiment_v3"
REPO, UNITS, DATA = HERE.parent, HERE.parent / "units", HERE / "data"
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
import m0_baseline as m0
from qfbench2_track_forecasting import scoring as S
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:.4g}")
WORKERS = max(1, (mp.cpu_count() or 2) - 2)
geo = lambda r: float(np.exp(np.log(np.asarray(r, float)).mean()))

ANSWERS = pd.read_parquet(DATA / "backtest_answers.parquet")
M0_SCORES = pd.read_parquet(DATA / "m0_backtest_scores.parquet")
M11_SCORES = pd.read_parquet(DATA / "m1_1_backtest_scores.parquet")
UNIT_IDS = sorted(ANSWERS.unit.unique())
RULES = tomllib.loads((HERE / "transformations.toml").read_text())
SH = RULES["shocks"]
WINDOW = 260

def asset_type(a):
    return RULES["assets"].get(a, "other")

def rule_for(asset, target_type):
    """card.toml's target type + the asset -> {transform, shock, drift} (transformations.toml)."""
    t = RULES["assets"].get(asset)
    base = RULES["types"][t] if t else {"transform": RULES["default"]["level"], "shock": RULES["default"]["shock"],
                                        "drift": RULES["default"]["drift"]}
    tr = RULES["default"]["log_return"] if target_type == "log_return" else base["transform"]
    return {"transform": tr, "shock": base["shock"], "drift": bool(base["drift"])}

def load_unit(u):
    ud = UNITS / u
    card = tomllib.loads((ud / "card.toml").read_text()); t = card["targets"]
    assets, horizons = [str(a) for a in t["asset_ids"]], [int(h) for h in t["horizons"]]
    uid, asof, tt = card["task"]["id"], card["provenance"]["data_cutoff"], t.get("target_type", "level")
    rows = {}
    for a in assets:
        for path in sorted(ud.glob("*.parquet")):
            tab = pq.read_table(path).to_pydict()
            acol = "asset" if "asset" in tab else ("asset_id" if "asset_id" in tab else None)
            if acol is None:
                continue
            r = sorted((str(d)[:10], float(v)) for d, x, v in zip(tab["date"], tab[acol], tab["value"])
                       if str(x) == a and str(d)[:10] <= asof and v is not None)
            if r:
                rows[a] = r; break
    return {"unit": u, "id": uid, "family": card["metadata"]["category"], "card": card, "assets": assets,
            "horizons": horizons, "target_type": tt, "asof": asof, "rows": rows, "dates": {a: [d for d, _ in rows[a]] for a in assets},
            "rules": {a: rule_for(a, tt) for a in assets}, "steps": {h: m0._MONTHLY_STEPS.get(uid, {}).get(h, h) for h in horizons}}

UNITS_DATA = {u: load_unit(u) for u in UNIT_IDS}
TYPE = {u: "+".join(sorted({asset_type(a) for a in U["assets"]})) for u, U in UNITS_DATA.items()}
BACKTEST = {}
for u, g in ANSWERS.groupby("unit"):
    U = UNITS_DATA[u]
    order = {(a, h): i for i, (a, h) in enumerate((a, h) for a in U["assets"] for h in U["horizons"])}
    BACKTEST[u] = [(o, gg.assign(i=[order[(a, h)] for a, h in zip(gg.asset, gg.horizon)]).sort_values("i").answer.to_numpy())
                   for o, gg in g.groupby("origin")]
print(f"{len(UNIT_IDS)} units, {sum(len(v) for v in BACKTEST.values()):,} backtest dates")
display(pd.DataFrame({t: {"transform": v["transform"], "shock": v["shock"], "drift": v["drift"]} for t, v in RULES["types"].items()}).T)
print("Student-t:", SH)

103 units, 23,635 backtest dates


,transform,shock,drift
rates,diff,student_t,False
fx,log_diff,student_t,False
cpi,log_diff,normal,True
payrolls,diff,normal,True
unemployment,diff,normal,False
equity_factor,as_return,student_t,False


Student-t: {'nu': 'fit', 'nu_min': 4.2, 'nu_max': 30.0, 'normal_below_kurtosis': 0.2}


## 1 — The model: notebook 07's M1.3 with an optional EWMA sd

`m13_draws(U, origin, ewma_halflife=63)` is M1.5; `ewma_halflife=None` is M1.3. Checks: (1) `None` reproduces notebook 07's
M1.3 scores; (2) one unit at the real as-of: the plain sd and the EWMA sd side by side.

In [2]:
def fit_nu(x, rows_t):
    """nu from the window's excess kurtosis k: a Student-t has k = 6 / (nu - 4), so nu = 4 + 6 / k."""
    if not rows_t:
        return None
    z = x[rows_t] - x[rows_t].mean(axis=1, keepdims=True)
    k = float(np.mean((z ** 4).mean(axis=1) / (z ** 2).mean(axis=1) ** 2 - 3))
    return None if k < SH["normal_below_kurtosis"] else float(np.clip(4 + 6 / k, SH["nu_min"], SH["nu_max"]))

def fit(U, origin, steps_window=None):
    """The window's steps per asset -> (assets sorted, steps matrix x, log mode, last value)."""
    by_asset = sorted(U["assets"]); steps, log_mode, last = {}, {}, {}
    for a in by_asset:
        k = bisect.bisect_right(U["dates"][a], origin)
        rows = U["rows"][a][max(0, k - (m0._WINDOW if steps_window is None else steps_window + 1)): k]
        log_mode[a] = U["rules"][a]["transform"] == "log_diff" and min(v for _, v in rows) > 0
        if log_mode[a]:
            steps[a], _ = m0._steps([(d, np.log(v)) for d, v in rows], "level"); last[a] = rows[-1][1]
        else:
            steps[a], lv = m0._steps(rows, U["target_type"]); last[a] = lv if U["target_type"] == "level" else 0.0
    common = sorted(set.intersection(*(set(steps[a]) for a in by_asset)))
    x = np.array([[steps[a][d] for d in common] for a in by_asset])
    return by_asset, x, log_mode, last

def ewma_sd_asset(U, a, origin, halflife):
    """EWMA sd of one asset's steps (its own transform) over the last 8 x halflife steps: weight 0.5 ** (age / halflife)."""
    k = bisect.bisect_right(U["dates"][a], origin)
    rows = U["rows"][a][max(0, k - (int(8 * halflife) + 1)): k]
    lm = U["rules"][a]["transform"] == "log_diff" and min(v for _, v in rows) > 0
    st, _ = m0._steps([(d, np.log(v)) for d, v in rows] if lm else rows, "level" if lm else U["target_type"])
    v = np.array([st[d] for d in sorted(st)])
    age = np.arange(len(v))[::-1]; w = 0.5 ** (age / halflife); w = w / w.sum()
    m = w @ v
    return float(np.sqrt(w @ (v - m) ** 2))

def m13_draws(U, origin, steps_window=WINDOW, drift_rule=True, student_t=True, ewma_halflife=None):
    """ewma_halflife=None: the plain sd of the window (M1.3); a number: the EWMA sd with that halflife (M1.5)."""
    by_asset, x, log_mode, last = fit(U, origin, steps_window)
    assets, horizons = U["assets"], U["horizons"]
    mu, sigma = x.mean(axis=1), np.atleast_2d(np.cov(x))
    if ewma_halflife is not None:                       # M1.5: EWMA sd per asset; correlation from the window
        sd_w = np.sqrt(np.diag(sigma)); corr = np.nan_to_num(sigma / np.outer(sd_w, sd_w), nan=0.0); np.fill_diagonal(corr, 1.0)
        sd_e = np.array([ewma_sd_asset(U, a, origin, ewma_halflife) for a in by_asset])
        sigma = corr * np.outer(sd_e, sd_e)
    ai = {a: i for i, a in enumerate(by_asset)}
    cells = [(a, h) for a in by_asset for h in sorted(horizons)]; s = U["steps"]
    def centre(a, h):
        anchor = 0.0 if log_mode[a] else last[a]
        use_drift = U["rules"][a]["drift"] if drift_rule else True
        return anchor + (s[h] * mu[ai[a]] if use_drift else 0.0)
    mean = np.array([centre(a, h) for a, h in cells])
    cov = np.array([[min(s[h1], s[h2]) * sigma[ai[a1], ai[a2]] for a2, h2 in cells] for a1, h1 in cells])
    cov[np.diag_indices_from(cov)] += 1e-10; j = cov + 1e-9 * np.eye(len(cells))
    try: f = np.linalg.cholesky(j)
    except np.linalg.LinAlgError: f = np.diag(np.sqrt(np.diag(j)))
    seed = zlib.crc32(U["id"].encode()) & 0x7FFFFFFF
    dev = np.random.default_rng(seed).standard_normal((m0.N_DRAWS, len(cells))) @ f.T
    t_cells = np.array([U["rules"][a]["shock"] == "student_t" for a, h in cells])
    nu = fit_nu(x, [ai[a] for a in by_asset if U["rules"][a]["shock"] == "student_t"]) if student_t else None
    if nu is not None and t_cells.any():
        dev = dev * np.where(t_cells, np.sqrt((nu - 2.0) / np.random.default_rng([seed, 7]).chisquare(nu, size=(m0.N_DRAWS, 1))), 1.0)
    sm = mean + dev
    out = np.empty((m0.N_DRAWS, len(assets), len(horizons)))
    for ci, (a, h) in enumerate(cells):
        col = last[a] * np.exp(sm[:, ci]) if log_mode[a] else sm[:, ci]
        if s[h] <= 0:
            col = np.full(m0.N_DRAWS, U["rows"][a][bisect.bisect_right(U["dates"][a], origin) - 1][1])
        out[:, assets.index(a), horizons.index(h)] = col
    return out


cats = ["marginal", "joint", "tail"]

def parts(card, samples, y):
    p = card.get("scoring", {}).get("params", {})
    return S._composite(samples, y, weights=m0.card_weights(card, samples.shape[1]),
                        tail_levels=tuple(p.get("tail_levels", (0.01, 0.05, 0.95, 0.99))), joint=S.card_joint_statistic(card),
                        tail_metric=str(p.get("tail_metric", S.DEFAULT_TAIL_METRIC)), ref_scale=None)
mean0 = M0_SCORES.groupby("unit")[cats].mean()
def weights(u):
    U = UNITS_DATA[u]; return m0.card_weights(U["card"], len(U["assets"]) * len(U["horizons"]))
def vs_m0(scores):
    m = scores.groupby("unit")[cats].mean()
    return pd.Series({u: sum(wk * m.loc[u, k] / mean0.loc[u, k] for k, wk in zip(cats, weights(u)) if wk) for u in UNIT_IDS})

U = UNITS_DATA["t2-F2-higher-for-longer-2023"]; a = U["assets"][0]
_, x, _, _ = fit(U, U["asof"], WINDOW)
print(f"{U['unit']} ({a}): plain sd of the 260 steps = {float(np.sqrt(np.cov(x))):.5f}, "
      f"EWMA sd (halflife 63) = {ewma_sd_asset(U, a, U['asof'], 63):.5f}")

def score_unit(u):
    U = UNITS_DATA[u]
    return [{"unit": u, "family": U["family"], "origin": o, "model": m, **parts(U["card"], m13_draws(U, o, ewma_halflife=hl).reshape(500, -1), y)}
            for o, y in BACKTEST[u] for m, hl in (("M1.3", None), ("M1.5", 63))]
t0 = time.time()
with mp.get_context("fork").Pool(WORKERS) as pool:
    G = pd.DataFrame([r for rows in pool.map(score_unit, UNIT_IDS, chunksize=1) for r in rows])
print(f"scored {len(G) // 2:,} dates x 2 models in {time.time() - t0:.0f}s")
d = float((vs_m0(G[G.model == "M1.3"]) - vs_m0(pd.read_parquet(DATA / "m1_3_backtest_scores.parquet"))).abs().max())
print(f"ewma_halflife=None vs notebook 07's saved M1.3, competition score per unit: max |difference| = {d:.1e}")
assert d < 1e-12
M15 = G[G.model == "M1.5"].drop(columns="model").reset_index(drop=True)

t2-F2-higher-for-longer-2023 (UST_10Y): plain sd of the 260 steps = 0.07692, EWMA sd (halflife 63) = 0.06960


scored 23,635 dates x 2 models in 4s
ewma_halflife=None vs notebook 07's saved M1.3, competition score per unit: max |difference| = 0.0e+00


## 2 — Every model on the same backtest

In [3]:
fam = pd.Series({u: UNITS_DATA[u]["family"] for u in UNIT_IDS})
_w12 = pd.read_parquet(DATA / "m1_2_window_backtest_scores.parquet")
models = {"M0": M0_SCORES, "M1.1 (per asset type)": M11_SCORES,
          "M1.2 (t fitted, 2 years, M0 drift)": _w12[_w12.window == 504].drop(columns="window"),
          "production (text off)": pd.read_parquet(DATA / "production_backtest_scores.parquet"),
          "M1.3 (plain sd, 260)": pd.read_parquet(DATA / "m1_3_backtest_scores.parquet"),
          "M1.4 (best nu per family)": pd.read_parquet(DATA / "m1_4_backtest_scores.parquet"),
          "M1.5 (EWMA sd, halflife 63)": M15}
SC = {name: vs_m0(sc) for name, sc in models.items()}
LADDER = pd.DataFrame({name: {**{f: s[fam[fam == f].index].mean() for f in sorted(fam.unique())}, "all": s.mean(),
                              "better than M0": f"{(s < 1).sum()}/{len(s)}"} for name, s in SC.items()}).T
LADDER

,T2-F1,T2-F2,T2-F3,T2-F4,all,better than M0
M0,1,1,1,1,1,0/103
M1.1 (per asset type),0.9955,0.999,0.9993,0.9968,0.9976,25/103
"M1.2 (t fitted, 2 years, M0 drift)",0.97,0.9967,0.9761,1.01,0.9902,64/103
production (text off),0.9288,1.011,0.9274,1.003,0.9726,74/103
"M1.3 (plain sd, 260)",0.8951,0.9995,0.9194,0.9763,0.9521,81/103
M1.4 (best nu per family),0.8929,0.999,0.9191,0.9769,0.9516,81/103
"M1.5 (EWMA sd, halflife 63)",0.8932,0.99,0.9141,0.9642,0.9444,85/103


## 3 — Per family and per asset type: M1.3 vs M1.5

In [4]:
m15_mean, m0_mean = M15.groupby("unit")[cats].mean(), M0_SCORES.groupby("unit")[cats].mean()
PER_UNIT = pd.DataFrame({"family": fam, "asset type": pd.Series(TYPE),
                         **{f"{n} ÷ M0 (M1.5)": pd.Series({u: (m15_mean.loc[u, k] / m0_mean.loc[u, k] if weights(u)[i] else np.nan) for u in UNIT_IDS})
                            for i, (k, n) in enumerate(zip(cats, ["CRPS", "variogram", "tail"]))},
                         "M1.3 vs M0": SC["M1.3 (plain sd, 260)"], "M1.5 vs M0": SC["M1.5 (EWMA sd, halflife 63)"],
                         "production vs M0": SC["production (text off)"]})
def summary(g):
    return {"units": len(g), "M1.3": g["M1.3 vs M0"].mean(), "M1.5": g["M1.5 vs M0"].mean(), "production": g["production vs M0"].mean(),
            "M1.5 better than M1.3": f'{(g["M1.5 vs M0"] < g["M1.3 vs M0"]).sum()}/{len(g)}',
            "M1.5 better than production": f'{(g["M1.5 vs M0"] < g["production vs M0"]).sum()}/{len(g)}'}
FAMILY = pd.DataFrame({f: summary(g) for f, g in [*PER_UNIT.groupby("family"), ("all", PER_UNIT)]}).T.rename_axis("family")
BY_TYPE = pd.DataFrame({t: summary(g) for t, g in PER_UNIT.groupby("asset type")}).T.rename_axis("asset type")
print("per family:"); display(FAMILY)
print("per asset type:"); display(BY_TYPE)
print("per unit:"); display(PER_UNIT.rename_axis("unit").reset_index().set_index(["family", "unit"]).sort_index())

per family:


,units,M1.3,M1.5,production,M1.5 better than M1.3,M1.5 better than production
family,,,,,,
T2-F1,23,0.8951,0.8932,0.9288,12/23,15/23
T2-F2,27,0.9995,0.99,1.011,18/27,18/27
T2-F3,22,0.9194,0.9141,0.9274,14/22,15/22
T2-F4,31,0.9763,0.9642,1.003,26/31,24/31
all,103,0.9521,0.9444,0.9726,70/103,72/103


per asset type:


,units,M1.3,M1.5,production,M1.5 better than M1.3,M1.5 better than production
asset type,,,,,,
cpi,2,0.8735,0.899,1.308,0/2,2/2
equity_factor,16,0.9758,0.9665,0.9935,12/16,13/16
fx,35,0.9685,0.9614,0.9819,22/35,25/35
fx+rates,5,0.9465,0.933,0.9429,4/5,3/5
payrolls,1,1,0.9682,1.624,1/1,1/1
rates,43,0.9323,0.9234,0.9285,31/43,27/43
unemployment,1,0.9875,1.024,1.036,0/1,1/1


per unit:


asset type  CRPS ÷ M0 (M1.5)  variogram ÷ M0 (M1.5)  tail ÷ M0 (M1.5)  M1.3 vs M0  M1.5 vs M0  production vs M0
family unit                                                                                                                                                 
T2-F1  t2-F1-ai-mom-2024                  equity_factor            0.7308                    NaN            0.9077      0.7878      0.7814              0.77
       t2-F1-aud-on-hold-2016                        fx            0.8469                 0.9932             1.173      0.9182      0.9559            0.9478
       t2-F1-cad-boc-2017                            fx            0.9585                 0.9566             1.054      0.9815      0.9769             1.037
       t2-F1-chf-highly-valued-2021                  fx            0.8274                 0.8641            0.9396      0.8872      0.8608            0.8065
       t2-F1-conflicting-texts-2024               rates            0.9688                  1.023            0.9225      0.9962      0.9757             1.026
       t2-F1-considerable-period-2003             rates            0.9757                 0.8274            0.9157      0.9263      0.9192            0.9141
       t2-F1-conundrum-2005                       rates            0.5781                 0.8829            0.6969      0.7077      0.6933            0.6256
       t2-F1-cpi-glidepath-2023                     cpi            0.9158                 0.8472            0.8145      0.8531      0.8749             1.434
       t2-F1-dkk-peg-2019                            fx            0.7578                    NaN             0.817      0.7668      0.7747            0.8197
       t2-F1-eur-range-2017                          fx            0.7053                  0.919            0.9553      0.7981      0.8195            0.8291
       t2-F1-fed-put-2019                 equity_factor             1.184                    NaN            0.9716       1.116       1.123             1.196
       t2-F1-greater-confidence-2024              rates            0.9057                  0.894            0.9044      0.9441      0.9019            0.9257
       t2-F1-hawkish-cut-2024                     rates            0.8442                 0.8994            0.9074      0.9135      0.8734            0.8902
       t2-F1-jpy-ycc-flex-2018                       fx            0.7846                  1.044             0.835      0.8435      0.8724            0.9337
       t2-F1-liftoff-telegraph-2015               rates            0.8956                 0.8176            0.9786      0.9131      0.8888            0.7836
       t2-F1-measured-pace-2004                   rates            0.4462                 0.7862            0.7341      0.6306      0.6058            0.5517
       t2-F1-normalization-2017                   rates             0.727                    NaN            0.8787      0.7657      0.7703             0.826
       t2-F1-patient-dropped-2015                 rates            0.8501                    NaN            0.9886      0.9078      0.8896             0.786
       t2-F1-pause-2006                           rates              1.37                  1.374            0.9672       1.298       1.291             1.372
       t2-F1-qe3-end-2014                         rates            0.7476                 0.9192            0.8501      0.8192      0.8196            0.8437
       t2-F1-sahm-watch-2024               unemployment              1.02                   1.12            0.8885      0.9875       1.024             1.036
       t2-F1-third-cut-2019                       rates             0.949                  1.331            0.7134       1.004       1.016             1.116
       t2-F1-ust-positioning-2018                 rates            0.7721                 0.9354            0.8425      0.8208      0.8352            0.8929
T2-F2  t2-F2-abenomics-2012                          fx            0.8888                    NaN            0.9034      

## 4 — Save

In [5]:
M15.to_parquet(DATA / "m1_5_backtest_scores.parquet", index=False)
PER_UNIT.to_csv(DATA / "m1_5_vs_m0_per_unit.csv"); FAMILY.to_csv(DATA / "m1_5_vs_m0_per_family.csv")
BY_TYPE.to_csv(DATA / "m1_5_vs_m0_by_asset_type.csv"); LADDER.to_csv(DATA / "all_models_vs_m0.csv")
for f in ("m1_5_backtest_scores.parquet", "m1_5_vs_m0_per_unit.csv", "m1_5_vs_m0_per_family.csv", "m1_5_vs_m0_by_asset_type.csv"):
    print(f"data/{f:<30} {(DATA / f).stat().st_size / 1e3:8.1f} kB")

data/m1_5_backtest_scores.parquet      867.1 kB
data/m1_5_vs_m0_per_unit.csv            14.6 kB
data/m1_5_vs_m0_per_family.csv           0.5 kB
data/m1_5_vs_m0_by_asset_type.csv        0.6 kB


## 5 — In short

In [6]:
for name in LADDER.index:
    print(f"  {name:<36} {LADDER.loc[name, 'all']:.4f}   (better than M0 on {LADDER.loc[name, 'better than M0']})")
a = FAMILY.loc["all"]
print(f"M1.5 vs M1.3: better on {a['M1.5 better than M1.3']} units; vs production: better on {a['M1.5 better than production']}")
print("per family M1.5 (M1.3): " + "; ".join(f"{f} {FAMILY.loc[f, 'M1.5']:.4f} ({FAMILY.loc[f, 'M1.3']:.4f})" for f in FAMILY.index if f != "all"))

  M0                                   1.0000   (better than M0 on 0/103)
  M1.1 (per asset type)                0.9976   (better than M0 on 25/103)
  M1.2 (t fitted, 2 years, M0 drift)   0.9902   (better than M0 on 64/103)
  production (text off)                0.9726   (better than M0 on 74/103)
  M1.3 (plain sd, 260)                 0.9521   (better than M0 on 81/103)
  M1.4 (best nu per family)            0.9516   (better than M0 on 81/103)
  M1.5 (EWMA sd, halflife 63)          0.9444   (better than M0 on 85/103)
M1.5 vs M1.3: better on 70/103 units; vs production: better on 72/103
per family M1.5 (M1.3): T2-F1 0.8932 (0.8951); T2-F2 0.9900 (0.9995); T2-F3 0.9141 (0.9194); T2-F4 0.9642 (0.9763)
